# TUP Detection — Tier-B Guard Comparison

**Authors:** Vargas Pech · Fernández Castillo · Ruiz Peña · Rejón Quintal  
**Date:** June 2026  
**Purpose:** Compares TUP Layer 1 + Sentinel v2 against publicly reported guards on Tier-B datasets.

---

| Dataset | Rows | Metric |
|---------|------|--------|
| deepset/prompt-injections | 662 | Overall Accuracy |
| Antijection | ~5,988 | Overall Accuracy |

> External baselines (Lakera Guard, Sentinel v2, ProtectAI DeBERTa, Llama Guard) are sourced from the  
> Antijection paper and Sentinel v2 HF README. They are **not** re-run here — visual comparison only.


In [ ]:
%pip install -q matplotlib pandas python-dotenv pyyaml

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from dotenv import load_dotenv

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(REPO_ROOT / ".env")

RESULTS_DIR = REPO_ROOT / "notebooks" / "data" / "external" / "results"
EXTERNAL_DIR = REPO_ROOT / "notebooks" / "data" / "external"
SCRIPTS = REPO_ROOT / "scripts"
sys.path.insert(0, str(REPO_ROOT / "tup-manager"))

PALETTE = {
    "ours": "#00F5A0",
    "lakera": "#6366F1",
    "sentinel_paper": "#22D3EE",
    "deberta": "#F59E0B",
    "llama_guard": "#A78BFA",
    "other": "#94A3B8",
    "bg": "#0F172A",
    "grid": "#334155",
    "text": "#E2E8F0",
}

plt.rcParams.update({
    "figure.facecolor": PALETTE["bg"],
    "axes.facecolor": PALETTE["bg"],
    "axes.edgecolor": PALETTE["grid"],
    "axes.labelcolor": PALETTE["text"],
    "text.color": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "grid.color": PALETTE["grid"],
    "font.size": 11,
})

BENCHMARKS = {
    "deepset": {
        "yaml": EXTERNAL_DIR / "deepset.yaml",
        "preset": "deepset",
        "result_json": RESULTS_DIR / "deepset-sentinel-v2-rerun.json",
        "fallback_json": RESULTS_DIR / "deepset-sentinel-hf.json",
    },
    "antijection": {
        "yaml": EXTERNAL_DIR / "antijection.yaml",
        "preset": "antijection",
        "result_json": RESULTS_DIR / "antijection-sentinel-hf.json",
        "fallback_json": None,
    },
}

ENGINE = "TUP Layer 1+Classifier"
print(f"Repo: {REPO_ROOT}")
print(f"Results: {RESULTS_DIR}")

## 1. Run Benchmarks (optional)

Set `NOTEBOOK_RUN_BENCHMARKS=1` in the environment to re-run.  
Requires `SENTINEL_API_KEY` + `HF_INFERENCE_ENDPOINT` in `.env`.  
Antijection may take **1–3 h**. By default, pre-computed results are loaded from `notebooks/data/external/results/`.


In [ ]:
RUN_BENCHMARKS = os.getenv("NOTEBOOK_RUN_BENCHMARKS", "0") == "1"


def run_benchmark(name: str, cfg: dict) -> Path:
    out = cfg["result_json"]
    if out.is_file() and not RUN_BENCHMARKS:
        print(f"[{name}] skip — exists {out.name}")
        return out
    yaml_path = cfg["yaml"]
    if not yaml_path.is_file():
        print(f"[{name}] importing {cfg['preset']}...")
        subprocess.run(
            [sys.executable, str(SCRIPTS / "import_external_dataset.py"),
             "--preset", cfg["preset"], "--out", str(yaml_path)],
            cwd=REPO_ROOT, check=True,
            env={**os.environ, "PYTHONPATH": f"{REPO_ROOT / 'tup-manager'}:{SCRIPTS}"},
        )
    print(f"[{name}] running benchmark → {out.name}")
    subprocess.run(
        [sys.executable, str(SCRIPTS / "run_pint_benchmark.py"),
         "--dataset", str(yaml_path),
         "--detection-mode", "benchmark",
         "--engines", ENGINE,
         "--results-out", str(out)],
        cwd=REPO_ROOT, check=True,
        env={**os.environ, "PYTHONPATH": f"{REPO_ROOT / 'tup-manager'}:{SCRIPTS}"},
    )
    return out


if RUN_BENCHMARKS:
    for name, cfg in BENCHMARKS.items():
        run_benchmark(name, cfg)
else:
    print("Set NOTEBOOK_RUN_BENCHMARKS=1 to ejecutar benchmarks desde el notebook.")
    print("Por defecto se cargan JSON pre-computados en notebooks/data/external/results/.")

In [ ]:
def load_result(path: Path, fallback: Path | None = None) -> dict | None:
    for p in [path, fallback]:
        if p and p.is_file():
            return json.loads(p.read_text(encoding="utf-8"))
    return None


def extract_engine_row(payload: dict, engine: str = ENGINE) -> dict:
    for row in payload.get("results", []):
        if row.get("engine") == engine:
            return row
    raise KeyError(f"Engine {engine!r} not in {payload.get('dataset')}")


ours = {}
for name, cfg in BENCHMARKS.items():
    data = load_result(cfg["result_json"], cfg.get("fallback_json"))
    if data:
        row = extract_engine_row(data)
        ours[name] = {
            "dataset": name,
            "rows": data.get("rows"),
            "overall_accuracy_pct": row["overall_accuracy_pct"],
            "pint_balanced_pct": row["pint_balanced_pct"],
            "attack_detection_pct": row["attack_detection_pct"],
            "benign_pass_pct": row["benign_pass_pct"],
            "scoring_seconds": data.get("scoring_seconds"),
            "api_calls": data.get("api_calls"),
            "model": data.get("injection_model"),
        }
    else:
        print(f"⚠ Missing results for {name}")

df_ours = pd.DataFrame(ours.values()) if ours else pd.DataFrame()
df_ours

## 2. Public Baseline References

Values reported by third parties: Overall Accuracy on Antijection; deepset per Sentinel v2 paper (F1 ≈ accuracy proxy).


In [ ]:
# Fuentes: Antijection paper/leaderboard, Sentinel v2 HF README, Qualifire SLM docs
PUBLIC_GUARDS = pd.DataFrame([
    {"guard": "Lakera Guard", "deepset_acc": np.nan, "antijection_acc": 97.0,
     "source": "Antijection benchmark (reported)", "color": PALETTE["lakera"]},
    {"guard": "Sentinel v2 (paper)", "deepset_acc": 88.0, "antijection_acc": 94.5,
     "source": "rogue-security README avg F1/deepset col", "color": PALETTE["sentinel_paper"]},
    {"guard": "ProtectAI DeBERTa v2", "deepset_acc": 77.6, "antijection_acc": 75.0,
     "source": "Our deepset run + Antijection ref", "color": PALETTE["deberta"]},
    {"guard": "Llama Guard 3 8B", "deepset_acc": np.nan, "antijection_acc": 62.8,
     "source": "Antijection / Qualifire SLM table", "color": PALETTE["llama_guard"]},
    {"guard": "Qwen3Guard 0.6B", "deepset_acc": np.nan, "antijection_acc": 85.8,
     "source": "Qualifire SLM table", "color": PALETTE["other"]},
])

# Nuestro stack
if not df_ours.empty:
    tup_row = {
        "guard": "TUP + Sentinel v2 (ours)",
        "deepset_acc": df_ours.loc[df_ours["dataset"] == "deepset", "overall_accuracy_pct"].squeeze() if "deepset" in ours else np.nan,
        "antijection_acc": df_ours.loc[df_ours["dataset"] == "antijection", "overall_accuracy_pct"].squeeze() if "antijection" in ours else np.nan,
        "source": "This repo benchmark",
        "color": PALETTE["ours"],
    }
    PUBLIC_GUARDS = pd.concat([PUBLIC_GUARDS, pd.DataFrame([tup_row])], ignore_index=True)

PUBLIC_GUARDS["avg_tier_b"] = PUBLIC_GUARDS[["deepset_acc", "antijection_acc"]].mean(axis=1, skipna=True)
PUBLIC_GUARDS.sort_values("avg_tier_b", ascending=False)

## 3. Comparative Analysis


In [ ]:
def bar_compare(dataset: str, col: str, title: str, ylabel: str = "Overall Accuracy (%)"):
    sub = PUBLIC_GUARDS.dropna(subset=[col]).sort_values(col, ascending=True)
    if sub.empty:
        print(f"No data for {dataset}")
        return
    fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(sub))))
    colors = sub["color"].tolist()
    bars = ax.barh(sub["guard"], sub[col], color=colors, edgecolor="white", linewidth=0.5)
    for bar, val in zip(bars, sub[col]):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                f"{val:.1f}%", va="center", fontsize=10, color=PALETTE["text"])
    ax.set_xlim(0, 102)
    ax.set_xlabel(ylabel)
    ax.set_title(title, fontsize=14, fontweight="bold", pad=12)
    ax.axvline(90, color=PALETTE["grid"], linestyle="--", alpha=0.6, label="90% target")
    ax.grid(axis="x", alpha=0.3)
    ours_val = sub.loc[sub["guard"].str.contains("ours", case=False), col]
    if len(ours_val):
        best_other = sub.loc[~sub["guard"].str.contains("ours", case=False), col].max()
        delta = ours_val.iloc[0] - best_other
        ax.text(0.02, 0.02, f"TUP vs best public: {delta:+.1f} pp",
                transform=ax.transAxes, fontsize=10, color=PALETTE["ours"])
    plt.tight_layout()
    plt.show()

bar_compare("deepset", "deepset_acc", "deepset (662 rows) — Overall Accuracy")
bar_compare("antijection", "antijection_acc", "Antijection (~6k rows) — Overall Accuracy")

In [ ]:
# Grouped bar: both Tier B datasets side by side
plot_df = PUBLIC_GUARDS.dropna(subset=["deepset_acc", "antijection_acc"], how="all").copy()
plot_df = plot_df.sort_values("avg_tier_b", ascending=False)

x = np.arange(len(plot_df))
w = 0.35
fig, ax = plt.subplots(figsize=(12, 6))
b1 = ax.bar(x - w/2, plot_df["deepset_acc"].fillna(0), w, label="deepset",
            color=[c if not np.isnan(v) else "#333" for c, v in zip(plot_df["color"], plot_df["deepset_acc"])],
            alpha=0.85)
b2 = ax.bar(x + w/2, plot_df["antijection_acc"].fillna(0), w, label="Antijection",
            color=[c if not np.isnan(v) else "#333" for c, v in zip(plot_df["color"], plot_df["antijection_acc"])],
            alpha=0.55, hatch="///")

for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.5, f"{h:.1f}", ha="center", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["guard"], rotation=25, ha="right")
ax.set_ylabel("Overall Accuracy (%)")
ax.set_ylim(0, 105)
ax.set_title("Tier B Head-to-Head: TUP + Sentinel v2 vs Public Guards", fontweight="bold")
ax.legend(loc="lower right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Radar / spider for our engines on available datasets
if not df_ours.empty:
    metrics = ["overall_accuracy_pct", "pint_balanced_pct", "attack_detection_pct", "benign_pass_pct"]
    labels = ["Overall Acc", "PINT Balanced", "Attack Recall", "Benign Pass"]

    fig, axes = plt.subplots(1, len(df_ours), figsize=(5 * len(df_ours), 5), subplot_kw=dict(polar=True))
    if len(df_ours) == 1:
        axes = [axes]
    angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]

    for ax, (_, row) in zip(axes, df_ours.iterrows()):
        vals = [row[m] for m in metrics] + [row[metrics[0]]]
        ax.plot(angles, vals, color=PALETTE["ours"], linewidth=2)
        ax.fill(angles, vals, color=PALETTE["ours"], alpha=0.25)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(labels, size=9)
        ax.set_ylim(0, 100)
        ax.set_title(row["dataset"].upper(), fontweight="bold", pad=20)
    plt.suptitle("TUP + Sentinel v2 — métricas por dataset", fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Ranking table
rank = PUBLIC_GUARDS.sort_values("avg_tier_b", ascending=False).reset_index(drop=True)
rank["rank"] = rank.index + 1
rank_display = rank[["rank", "guard", "deepset_acc", "antijection_acc", "avg_tier_b", "source"]]
rank_display.columns = ["#", "Guard", "deepset %", "Antijection %", "Avg Tier B %", "Source"]

fig, ax = plt.subplots(figsize=(14, 0.45 * len(rank_display) + 1.5))
ax.axis("off")
tbl = ax.table(
    cellText=rank_display.round(1).values,
    colLabels=rank_display.columns,
    loc="center",
    cellLoc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.4)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor("#1E293B")
        cell.set_text_props(color="white", fontweight="bold")
    elif r > 0 and "ours" in str(rank_display.iloc[r-1]["Guard"]).lower():
        cell.set_facecolor("#064E3B")
ax.set_title("Tier B Leaderboard (Overall Accuracy)", fontweight="bold", pad=20)
plt.show()

## 4. Conclusión automática

In [ ]:
if "deepset" in ours:
    d = ours["deepset"]["overall_accuracy_pct"]
    print(f"deepset: {d:.2f}% overall accuracy")
if "antijection" in ours:
    a = ours["antijection"]["overall_accuracy_pct"]
    print(f"Antijection: {a:.2f}% overall accuracy")

if not rank.empty:
    top = rank.iloc[0]
    ours_rank = rank[rank["guard"].str.contains("ours", case=False)]
    if len(ours_rank):
        r = int(ours_rank.iloc[0]["rank"])
        print(f"\n🏁 TUP + Sentinel v2 rank: #{r} of {len(rank)} (avg Tier B)")
        if r == 1:
            print("✅ #1 en promedio Tier B vs referencias públicas cargadas.")
        elif len(ours_rank) and ours_rank.iloc[0]["avg_tier_b"] >= 95:
            print("✅ Dentro del rango top-tier (≥95% avg). Comparar con Lakera en Antijection.")